In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import random
from glob import glob
from tqdm import tqdm

processed_dir = r'Z:\image_enhancer\data\processed'
raw_dir = r'Z:\image_enhancer\data\raw'
soft_dir = r'Z:\image_enhancer\data\soft_dataset'

# Создаём папки
for split in ['train', 'test']:
    for category in ['original', 'corrupted']:
        path = os.path.join(soft_dir, split, category)
        os.makedirs(path, exist_ok=True)

print("Папки созданы")

# Функция мягкой порчи
def corrupt_soft(img):
    """
    Умеренная порча: яркость, контраст, насыщенность снижаются несильно
    """
    img_resized = cv2.resize(img, (128, 128))
    
    # Мягкие параметры (не перекручиваем)
    brightness = random.uniform(0.6, 0.95)
    contrast = random.uniform(0.6, 0.95)
    saturation = random.uniform(0.4, 0.85)
    
    # Применяем яркость и контраст
    img_corrupted = cv2.convertScaleAbs(
        img_resized,
        alpha=contrast,
        beta=128 * (1 - contrast) + (brightness - 1) * 128
    )
    
    # Применяем насыщенность
    hsv = cv2.cvtColor(img_corrupted, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:, :, 1] = hsv[:, :, 1] * saturation
    hsv[:, :, 1] = np.clip(hsv[:, :, 1], 0, 255)
    img_corrupted = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    
    # Коэффициенты для восстановления (нормализованы)
    k_brightness = np.clip((1.0 / brightness - 0.5) / 1.5, 0, 1)
    k_contrast = np.clip((1.0 / contrast - 0.5) / 1.5, 0, 1)
    k_saturation = np.clip((1.0 / saturation) / 2.0, 0, 1)
    
    return img_resized, img_corrupted, [k_brightness, k_contrast, k_saturation]

# Загружаем все фото из raw
image_files = []
for ext in ['*.jpg', '*.jpeg', '*.png']:
    image_files.extend(glob(os.path.join(raw_dir, '**', ext), recursive=True))

print(f"Найдено {len(image_files)} фото")

random.shuffle(image_files)
split_idx = int(len(image_files) * 0.85)
train_files = image_files[:split_idx]
test_files = image_files[split_idx:]

print(f"Train: {len(train_files)}, Test: {len(test_files)}")

records = []

# Обработка train
for idx, img_path in enumerate(tqdm(train_files, desc="Train")):
    img = cv2.imread(img_path)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    orig, corr, coeffs = corrupt_soft(img_rgb)
    
    name = f"img_{idx:06d}.jpg"
    
    cv2.imwrite(os.path.join(soft_dir, 'train', 'original', name), 
                cv2.cvtColor(orig, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(soft_dir, 'train', 'corrupted', name), 
                cv2.cvtColor(corr, cv2.COLOR_RGB2BGR))
    
    records.append({
        'filename': name,
        'split': 'train',
        'k_brightness': coeffs[0],
        'k_contrast': coeffs[1],
        'k_saturation': coeffs[2]
    })

# Обработка test
for idx, img_path in enumerate(tqdm(test_files, desc="Test")):
    img = cv2.imread(img_path)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    orig, corr, coeffs = corrupt_soft(img_rgb)
    
    name = f"img_{idx + split_idx:06d}.jpg"
    
    cv2.imwrite(os.path.join(soft_dir, 'test', 'original', name), 
                cv2.cvtColor(orig, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(soft_dir, 'test', 'corrupted', name), 
                cv2.cvtColor(corr, cv2.COLOR_RGB2BGR))
    
    records.append({
        'filename': name,
        'split': 'test',
        'k_brightness': coeffs[0],
        'k_contrast': coeffs[1],
        'k_saturation': coeffs[2]
    })

df = pd.DataFrame(records)
df.to_csv(os.path.join(soft_dir, 'labels.csv'), index=False)

print(f"Готово. Создано {len(records)} пар")

Папки созданы
Найдено 4319 фото
Train: 3671, Test: 648


Test: 100%|██████████████████████████████████████████████████████████████████████████| 648/648 [00:10<00:00, 63.56it/s]

Готово. Создано 4319 пар


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import cv2
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm

soft_dir = r'Z:\image_enhancer\data\soft_dataset'
df = pd.read_csv(os.path.join(soft_dir, 'labels.csv'))

print(f"Записей: {len(df)}")

def load_data(df, soft_dir):
    X = []
    y = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        split = row['split']
        filename = row['filename']
        img_path = os.path.join(soft_dir, split, 'corrupted', filename)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        coeffs = np.array([row['k_brightness'], row['k_contrast'], row['k_saturation']])
        X.append(img)
        y.append(coeffs)
    return np.array(X), np.array(y)

X, y = load_data(df, soft_dir)
print(f"Загружено: {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

def create_model():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(256, (3,3), activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(3, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = create_model()
model.summary()

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=20,
    batch_size=32,
    verbose=1
)

test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}")

# Сохраняем
model.save(r'Z:\image_enhancer\models\enhancer_model_soft.keras')
print("Модель сохранена")

Записей: 4319


100%|█████████████████████████████████████████████████████████████████████████████| 4319/4319 [00:30<00:00, 141.27it/s]


Загружено: (4319, 128, 128, 3)
Train: 3671, Test: 648


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 12, 12, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 421,699 (1.61 MB)

 Trainable params: 421,699 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 32s 309ms/step - loss: 0.0179 - mae: 0.1127 - val_loss: 0.0143 - val_mae: 0.1018
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 23s 239ms/step - loss: 0.0149 - mae: 0.1028 - val_loss: 0.0134 - val_mae: 0.0980
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 22s 224ms/step - loss: 0.0141 - mae: 0.0997 - val_loss: 0.0139 - val_mae: 0.1001
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 40s 209ms/step - loss: 0.0134 - mae: 0.0968 - val_loss: 0.0124 - val_mae: 0.0933
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 20s 203ms/step - loss: 0.0128 - mae: 0.0938 - val_loss: 0.0122 - val_mae: 0.0907
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 21s 206ms/step - loss: 0.0122 - mae: 0.0907 - val_loss: 0.0121 - val_mae: 0.0909
Epoch 7/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 22s 221ms/step - loss: 0.0119 - mae: 0.0894 - val_loss: 0.0113 - val_mae: 0.0872
Epoch 8/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 22s 220ms/step - loss: 0.0114 - mae: 0.0873 - val_loss: 0.0110 - val_mae: 0.0854
Epoch 9/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 21s 215ms/